In [0]:
COPY INTO market_risk_dev.bronze.market_prices
FROM (
  SELECT
    CAST(Date AS DATE) AS Date,
    CAST(Open AS DOUBLE) AS Open,
    CAST(High AS DOUBLE) AS High,
    CAST(Low AS DOUBLE) AS Low,
    CAST(Close AS DOUBLE) AS Close,
    CAST(Volume AS DOUBLE) AS Volume,
    CAST(ticker AS STRING)    AS ticker,
    CAST(fetched_at AS STRING) AS fetched_at,
    _metadata.file_path  AS _source_file,
    current_timestamp()  AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/prices/'
)
FILEFORMAT = CSV
PATTERN = 'year=*/month=*/day=*/*.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);

In [0]:
COPY INTO market_risk_dev.bronze.positions
FROM (
  SELECT
    trade_id,
    desk,
    book_id,
    trader_id,
    counterparty,
    asset_class,
    instrument_type,
    ticker,
    isin,
    cusip,
    direction,
    CAST(notional AS DOUBLE)    AS notional,
    currency,
    CAST(trade_date AS DATE)    AS trade_date,
    CAST(maturity_date AS DATE) AS maturity_date,
    CAST(generated_at AS STRING) AS generated_at,
    _metadata.file_path         AS _source_file,
    current_timestamp()         AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/positions/'
)
FILEFORMAT = CSV
PATTERN = 'year=*/month=*/day=*/*.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);

In [0]:
COPY INTO market_risk_dev.bronze.desk_limits
FROM (
  SELECT
    desk,
    CAST(limit_usd AS DOUBLE)        AS limit_usd,
    limit_currency,
    CAST(effective_date AS DATE)     AS effective_date,
    CAST(review_date AS DATE)        AS review_date,
    CAST(generated_at AS STRING)       AS generated_at,
    _metadata.file_path              AS _source_file,
    current_timestamp()              AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/reference/'
)
FILEFORMAT = CSV
PATTERN = 'desk_limits.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'force' = 'true'
);

In [0]:

COPY INTO market_risk_dev.bronze.fx_rates
FROM (
  SELECT
    currency,
    CAST(rate_vs_usd AS DOUBLE) AS rate_vs_usd,
    CAST(as_of_date AS DATE)    AS as_of_date,
    CAST(generated_at AS STRING) AS generated_at,
    _metadata.file_path         AS _source_file,
    current_timestamp()         AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/reference/'
)
FILEFORMAT = CSV
PATTERN = 'fx_rates.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'force' = 'true'
);

In [0]:
SELECT "market_prices" AS table_name, fetched_at, count(1) AS row_count 
  FROM market_risk_dev.bronze.market_prices
  group by fetched_at
 UNION ALL
SELECT "positions", generated_at, count(1) 
  FROM market_risk_dev.bronze.positions
 GROUP BY generated_at
 UNION ALL
SELECT "desk_limits", generated_at, count(1) 
  FROM market_risk_dev.bronze.desk_limits
 GROUP BY generated_at
 UNION ALL
SELECT "fx_rates", generated_at, count(1) 
  FROM market_risk_dev.bronze.fx_rates
GROUP BY generated_at
ORDER BY 1;